# Search Combination Evaluation

Compare retrieval performance for three combinations against the chunk-level ground truth:

- **3 combined**: vector + keyword + graph
- **no graph**: vector + keyword
- **vector and graph**: vector + graph

The ground truth file points each question to an expected Chroma chunk. Because graph results are facts rather than chunks, chunk metrics only count document chunks. Document metrics also check whether any returned item points to the expected document.

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path
from time import perf_counter
from typing import Any

from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.rag.constants import (  # noqa: E402
    DEFAULT_TOP_K,
    GRAPH_EMBEDDING_MODEL_NAME,
    RETRIEVAL_CANDIDATE_MULTIPLIER,
)
from backend.rag.helpers import load_cross_encoder  # noqa: E402
from backend.rag.search.graph_search import graph_search  # noqa: E402
from backend.rag.search.hybrid_document_search import merge_document_results  # noqa: E402
from backend.rag.search.keyword_search import keyword_search  # noqa: E402
from backend.rag.search.vector_search import load_collection, search as vector_search  # noqa: E402

load_dotenv(PROJECT_ROOT / ".env", override=True)

GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"
TOP_K = DEFAULT_TOP_K
CANDIDATE_K = max(TOP_K, TOP_K * RETRIEVAL_CANDIDATE_MULTIPLIER)

# Keep this small while experimenting with graph search. Set to None for all questions.
MAX_QUESTIONS = 25

COMBINATIONS = {
    "3 combined": {"vector": True, "keyword": True, "graph": True},
    "no graph": {"vector": True, "keyword": True, "graph": False},
    "vector and graph": {"vector": True, "keyword": False, "graph": True},
}

if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Ground truth: {GROUND_TRUTH_PATH}")
print(f"TOP_K={TOP_K} | CANDIDATE_K={CANDIDATE_K} | MAX_QUESTIONS={MAX_QUESTIONS}")

Project root: /workspace
Ground truth: /workspace/backend/data/eval/ground_truth_chunk_questions.jsonl
TOP_K=5 | CANDIDATE_K=20 | MAX_QUESTIONS=25


In [2]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_document_id(value: Any) -> str:
    text = str(value or "").strip()
    if not text:
        return ""
    text = Path(text).stem if "/" in text or text.endswith(".pdf") else text
    text = re.sub(r"-p\d{3}(?:-c\d{4})?(?:-g\d{4})?$", "", text)
    return text


def document_id_from_row(row: dict[str, Any]) -> str:
    metadata = row.get("metadata") or {}
    if not isinstance(metadata, dict):
        metadata = {}

    cypher_row = metadata.get("cypher_row") or {}
    if not isinstance(cypher_row, dict):
        cypher_row = {}

    candidates = [
        row.get("document_id"),
        metadata.get("document_id"),
        metadata.get("source_id"),
        metadata.get("filename"),
        metadata.get("source_path"),
        row.get("chunk_id"),
    ]
    candidates.extend(
        value
        for key, value in cypher_row.items()
        if "document" in str(key).lower() or str(key).lower().endswith("_id")
    )
    candidates.extend(row.get("triplet") or [])

    for candidate in candidates:
        document_id = normalize_document_id(candidate)
        if document_id:
            return document_id
    return ""


def triplet_text(triplet: Any) -> str:
    if isinstance(triplet, (list, tuple)) and len(triplet) == 3:
        return " -> ".join(str(part).strip() for part in triplet)
    return ""


def mark_document_rows(rows: list[dict[str, Any]], method: str) -> list[dict[str, Any]]:
    marked = []
    for rank, row in enumerate(rows, start=1):
        methods = list(row.get("retrieval_methods") or [])
        if method not in methods:
            methods.append(method)
        marked.append({**row, "kind": "document", "retrieval_methods": methods, f"{method}_rank": rank})
    return marked


def rerank_mixed_context(
    *,
    query: str,
    document_results: list[dict[str, Any]],
    graph_results: list[dict[str, Any]],
    top_k: int,
) -> list[dict[str, Any]]:
    candidates: list[dict[str, Any]] = []
    for row in document_results:
        candidates.append({**row, "kind": "document", "rerank_text": row.get("text", "")})
    for row in graph_results:
        text = f"{triplet_text(row.get('triplet'))}\n{row.get('text', '')}".strip()
        candidates.append({**row, "kind": "graph", "rerank_text": text})

    if not candidates:
        return []

    cross_encoder = load_cross_encoder()
    scores = cross_encoder.predict([(query, row.get("rerank_text", "")) for row in candidates])
    for row, score in zip(candidates, scores, strict=False):
        row["rerank_score"] = float(score)

    ranked = sorted(candidates, key=lambda row: row.get("rerank_score", float("-inf")), reverse=True)
    return [{**row, "rank": rank} for rank, row in enumerate(ranked[:top_k], start=1)]


def retrieve_context(
    *,
    query: str,
    combination: dict[str, bool],
    openai_client: Any,
    collection: Any,
) -> tuple[list[dict[str, Any]], str | None]:
    vector_rows: list[dict[str, Any]] = []
    keyword_rows: list[dict[str, Any]] = []
    graph_rows: list[dict[str, Any]] = []
    graph_error = None

    if combination.get("vector"):
        vector_rows = mark_document_rows(
            vector_search(openai_client=openai_client, collection=collection, query=query, top_k=CANDIDATE_K),
            "vector",
        )

    if combination.get("keyword"):
        keyword_rows = mark_document_rows(keyword_search(collection=collection, query=query, top_k=CANDIDATE_K), "keyword")

    if vector_rows and keyword_rows:
        document_rows = merge_document_results(vector_results=vector_rows, keyword_results=keyword_rows)
    else:
        document_rows = vector_rows or keyword_rows

    if combination.get("graph"):
        try:
            graph_rows = graph_search(query=query, embedding_model=GRAPH_EMBEDDING_MODEL_NAME)
        except Exception as exc:
            graph_error = str(exc)
            graph_rows = []

    return rerank_mixed_context(query=query, document_results=document_rows, graph_results=graph_rows, top_k=TOP_K), graph_error


def retrieved_item_summary(row: dict[str, Any]) -> dict[str, Any]:
    return {
        "rank": row.get("rank"),
        "kind": row.get("kind"),
        "chunk_id": row.get("chunk_id"),
        "document_id": document_id_from_row(row),
        "methods": ", ".join(row.get("retrieval_methods") or []),
        "score": row.get("rerank_score"),
        "text": " ".join(str(row.get("text") or "").split())[:180],
    }


def rank_of_expected_chunk(rows: list[dict[str, Any]], expected_chunk_id: str) -> int | None:
    for row in rows:
        if row.get("chunk_id") == expected_chunk_id:
            return int(row["rank"])
    return None


def rank_of_expected_document(rows: list[dict[str, Any]], expected_document_id: str) -> int | None:
    expected = normalize_document_id(expected_document_id)
    for row in rows:
        if expected and document_id_from_row(row) == expected:
            return int(row["rank"])
    return None


def score_question(
    *,
    combination_name: str,
    ground_truth: dict[str, Any],
    rows: list[dict[str, Any]],
    elapsed_seconds: float,
    graph_error: str | None,
) -> dict[str, Any]:
    chunk_rank = rank_of_expected_chunk(rows, ground_truth["expected_chunk_id"])
    document_rank = rank_of_expected_document(rows, ground_truth.get("expected_document_id"))
    return {
        "combination": combination_name,
        "question_id": ground_truth.get("question_id"),
        "question": ground_truth.get("question"),
        "expected_chunk_id": ground_truth.get("expected_chunk_id"),
        "expected_document_id": ground_truth.get("expected_document_id"),
        "chunk_hit": int(chunk_rank is not None),
        "chunk_rank": chunk_rank,
        "chunk_reciprocal_rank": 0.0 if chunk_rank is None else 1.0 / chunk_rank,
        "document_hit": int(document_rank is not None),
        "document_rank": document_rank,
        "graph_items": sum(1 for row in rows if row.get("kind") == "graph"),
        "graph_error": graph_error,
        "elapsed_seconds": elapsed_seconds,
        "retrieved_items": [retrieved_item_summary(row) for row in rows],
    }

In [3]:
ground_truth_rows = load_jsonl(GROUND_TRUTH_PATH)
if not ground_truth_rows:
    raise ValueError("Ground truth file is empty. Run notebooks/generate_ground_truth.ipynb first.")

if MAX_QUESTIONS is not None:
    ground_truth_rows = ground_truth_rows[:MAX_QUESTIONS]

openai_client, collection = load_collection()

print(f"Loaded {len(ground_truth_rows)} questions")
print(f"Chroma collection contains {collection.count()} chunks")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded 25 questions
Chroma collection contains 93 chunks


In [4]:
details: list[dict[str, Any]] = []

for combination_name, combination in COMBINATIONS.items():
    print("=" * 80)
    print(f"Evaluating {combination_name}: {combination}")

    for index, item in enumerate(ground_truth_rows, start=1):
        start = perf_counter()
        rows, graph_error = retrieve_context(
            query=item["question"],
            combination=combination,
            openai_client=openai_client,
            collection=collection,
        )
        elapsed = perf_counter() - start
        details.append(
            score_question(
                combination_name=combination_name,
                ground_truth=item,
                rows=rows,
                elapsed_seconds=elapsed,
                graph_error=graph_error,
            )
        )

        if index % 5 == 0 or index == len(ground_truth_rows):
            print(f"{index}/{len(ground_truth_rows)} done")

print(f"Scored {len(details)} question/combination pairs")

Evaluating 3 combined: {'vector': True, 'keyword': True, 'graph': True}


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
/workspace/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/.venv/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'libc10_cuda.so: cannot open shared object file: No such file or directory'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/workspace/.venv/lib/python3.11/site-packages/torchvision/datapoints/__ini

5/25 done
10/25 done
15/25 done
20/25 done
25/25 done
Evaluating no graph: {'vector': True, 'keyword': True, 'graph': False}
5/25 done
10/25 done
15/25 done
20/25 done
25/25 done
Evaluating vector and graph: {'vector': True, 'keyword': False, 'graph': True}
5/25 done
10/25 done
15/25 done
20/25 done
25/25 done
Scored 75 question/combination pairs


In [5]:
try:
    import pandas as pd

    details_df = pd.DataFrame(details)
    summary_df = (
        details_df.groupby("combination", as_index=False)
        .agg(
            questions=("question_id", "count"),
            chunk_hit_rate=("chunk_hit", "mean"),
            chunk_mrr=("chunk_reciprocal_rank", "mean"),
            document_hit_rate=("document_hit", "mean"),
            avg_graph_items=("graph_items", "mean"),
            graph_errors=("graph_error", lambda values: sum(value is not None for value in values)),
            avg_latency_seconds=("elapsed_seconds", "mean"),
        )
        .sort_values(["chunk_hit_rate", "chunk_mrr", "document_hit_rate"], ascending=False)
        .reset_index(drop=True)
    )

    display(summary_df)
except Exception:
    print(json.dumps(details[:3], indent=2))

,combination,questions,chunk_hit_rate,chunk_mrr,document_hit_rate,avg_graph_items,graph_errors,avg_latency_seconds
0,no graph,25,0.92,0.636667,0.92,0.00,0,4.212032
1,vector and graph,25,0.92,0.620000,0.92,1.00,0,4.094626
2,3 combined,25,0.92,0.590000,0.92,0.96,0,8.981013


In [6]:
# Most useful failures to inspect first.
misses_df = details_df[details_df["chunk_hit"] == 0].copy()
display(
    misses_df[
        [
            "combination",
            "question_id",
            "question",
            "expected_chunk_id",
            "expected_document_id",
            "document_hit",
            "graph_items",
            "graph_error",
        ]
    ].head(15)
)

,combination,question_id,question,expected_chunk_id,expected_document_id,document_hit,graph_items,graph_error
17,3 combined,certificate_016-p002-c0003-q03,What is the purpose of the document as stated ...,certificate_016-p002-c0003,certificate_016,0,0,None
21,3 combined,donation_money_006-p001-c0000-q02,Who is the notary that executed the deed?,donation_money_006-p001-c0000,donation_money_006,0,0,None
42,no graph,certificate_016-p002-c0003-q03,What is the purpose of the document as stated ...,certificate_016-p002-c0003,certificate_016,0,0,None
46,no graph,donation_money_006-p001-c0000-q02,Who is the notary that executed the deed?,donation_money_006-p001-c0000,donation_money_006,0,0,None
67,vector and graph,certificate_016-p002-c0003-q03,What is the purpose of the document as stated ...,certificate_016-p002-c0003,certificate_016,0,0,None
71,vector and graph,donation_money_006-p001-c0000-q02,Who is the notary that executed the deed?,donation_money_006-p001-c0000,donation_money_006,0,0,None


In [9]:
# Inspect one question across all combinations.
# Set QUESTION_ID to a specific id, or leave it as None to inspect the first miss.
QUESTION_ID = "donation_money_006-p001-c0000-q02" #"certificate_016-p002-c0003-q03"

if QUESTION_ID is None:
    selected_question_id = misses_df.iloc[0]["question_id"] if len(misses_df) else details_df.iloc[0]["question_id"]
else:
    selected_question_id = QUESTION_ID

selected = details_df[details_df["question_id"] == selected_question_id]
print(selected.iloc[0]["question"])
print(f"Expected chunk: {selected.iloc[0]['expected_chunk_id']}")
print(f"Expected document: {selected.iloc[0]['expected_document_id']}")

for _, row in selected.iterrows():
    print("\n" + "=" * 80)
    print(row["combination"])
    display(pd.DataFrame(row["retrieved_items"]))

Who is the notary that executed the deed?
Expected chunk: donation_money_006-p001-c0000
Expected document: donation_money_006

3 combined


,rank,kind,chunk_id,document_id,methods,score,text
0,1,document,donation_money_009-p002-c0004,donation_money_009,"vector, keyword",5.492123,"**SIGNATURES** In witness whereof, the parties..."
1,2,document,sale_015-p004-c0006,sale_015,"vector, keyword",4.737529,"Ghent, Belgium --- This deed is executed in tr..."
2,3,document,donation_money_007-p003-c0004,donation_money_007,"vector, keyword",4.435990,**NOTARY:** _________________________ Meester ...
3,4,document,donation_money_009-p001-c0000,donation_money_009,vector,2.907612,**NOTARIAL DEED OF DONATION** *Document ID: do...
4,5,document,purchase_003-p003-c0005,purchase_003,vector,2.865283,"**[Signature of Meester Pieter Van Renterghem,..."



no graph


,rank,kind,chunk_id,document_id,methods,score,text
0,1,document,donation_money_009-p002-c0004,donation_money_009,"vector, keyword",5.492123,"**SIGNATURES** In witness whereof, the parties..."
1,2,document,sale_015-p004-c0006,sale_015,"vector, keyword",4.737529,"Ghent, Belgium --- This deed is executed in tr..."
2,3,document,donation_money_007-p003-c0004,donation_money_007,"vector, keyword",4.435990,**NOTARY:** _________________________ Meester ...
3,4,document,donation_money_009-p001-c0000,donation_money_009,vector,2.907612,**NOTARIAL DEED OF DONATION** *Document ID: do...
4,5,document,purchase_003-p003-c0005,purchase_003,vector,2.865283,"**[Signature of Meester Pieter Van Renterghem,..."



vector and graph


,rank,kind,chunk_id,document_id,methods,score,text
0,1,document,donation_money_009-p002-c0004,donation_money_009,vector,5.492123,"**SIGNATURES** In witness whereof, the parties..."
1,2,document,sale_015-p004-c0006,sale_015,vector,4.737529,"Ghent, Belgium --- This deed is executed in tr..."
2,3,document,donation_money_007-p003-c0004,donation_money_007,vector,4.435990,**NOTARY:** _________________________ Meester ...
3,4,document,donation_money_009-p001-c0000,donation_money_009,vector,2.907612,**NOTARIAL DEED OF DONATION** *Document ID: do...
4,5,document,purchase_003-p003-c0005,purchase_003,vector,2.865283,"**[Signature of Meester Pieter Van Renterghem,..."
